In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
from sklearn.metrics import roc_auc_score

PROCESSED = Path("../data/processed")
REPORTS = Path("reports")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

In [2]:
cc = pd.read_parquet(PROCESSED / "credit_card_balance.parquet")
train_target = pd.read_parquet(
    PROCESSED / "application_train_reduced.parquet",
    columns=["SK_ID_CURR", "TARGET"]
)

print(f"credit_card_balance: {cc.shape}")
print(f"Memoria: {cc.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nColumnas: {list(cc.columns)}")
print(f"\nHead:")
print(cc.head(3))

credit_card_balance: (3840312, 23)
Memoria: 652.3 MB

Columnas: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']

Head:
   SK_ID_PREV  SK_ID_CURR  MONTHS_BALANCE  AMT_BALANCE  \
0     2562384      378907              -6       56.970   
1     2582071      363914              -1    63975.555   
2     1740877      371185              -7    31815.225   

   AMT_CREDIT_LIMIT_ACTUAL  AMT_DRAWINGS_ATM_CURRENT  AMT_DRAWINGS_CURRENT  \
0                   135000                       0.0                 877.5   
1     

In [3]:
n_curr = cc["SK_ID_CURR"].nunique()
n_prev = cc["SK_ID_PREV"].nunique()
train_with_cc = train_target["SK_ID_CURR"].isin(cc["SK_ID_CURR"]).sum()

print(f"SK_ID_CURR únicos: {n_curr:,}")
print(f"SK_ID_PREV únicos: {n_prev:,}")
print(f"Clientes train con tarjeta: {train_with_cc:,} "
      f"({100*train_with_cc/len(train_target):.2f}%)")
print(f"\nMONTHS_BALANCE rango: [{cc['MONTHS_BALANCE'].min()}, {cc['MONTHS_BALANCE'].max()}]")

rows_per_client = cc.groupby("SK_ID_CURR").size()
print(f"\n=== Filas mensuales por cliente ===")
print(rows_per_client.describe().round(1))

SK_ID_CURR únicos: 103,558
SK_ID_PREV únicos: 104,307
Clientes train con tarjeta: 86,905 (28.26%)

MONTHS_BALANCE rango: [-96, -1]

=== Filas mensuales por cliente ===
count    103558.0
mean         37.1
std          33.5
min           1.0
25%          10.0
50%          22.0
75%          75.0
max         192.0
dtype: float64


In [4]:
print("=== NAME_CONTRACT_STATUS ===")
vc = cc["NAME_CONTRACT_STATUS"].value_counts(dropna=False)
print(pd.DataFrame({"count": vc, "pct": (vc/len(cc)*100).round(3)}))

print("\n=== Numéricas clave ===")
key_num = ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT",
           "AMT_PAYMENT_CURRENT", "AMT_TOTAL_RECEIVABLE", "SK_DPD", "SK_DPD_DEF"]
print(cc[key_num].describe().round(1))

print("\n=== % nulos ordenado ===")
print((cc.isna().mean() * 100).round(2).sort_values(ascending=False).head(15))

=== NAME_CONTRACT_STATUS ===
                        count     pct
NAME_CONTRACT_STATUS                 
Active                3698436  96.306
Completed              128918   3.357
Signed                  11058   0.288
Demand                   1365   0.036
Sent proposal             513   0.013
Refused                    17   0.000
Approved                    5   0.000

=== Numéricas clave ===
       AMT_BALANCE  AMT_CREDIT_LIMIT_ACTUAL  AMT_DRAWINGS_CURRENT  \
count    3840312.0                3840312.0             3840312.0   
mean       58300.2                 153808.0                7433.4   
std       106307.0                 165145.7               33846.1   
min      -420250.2                      0.0               -6211.6   
25%            0.0                  45000.0                   0.0   
50%            0.0                 112500.0                   0.0   
75%        89046.7                 180000.0                   0.0   
max      1505902.2                1350000.0         

In [5]:
# Utilización = saldo / límite (la métrica reina de tarjetas)
cc["CC_UTILIZATION"] = (
    cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
).astype("float32")

overdraft_mask = (cc["AMT_CREDIT_LIMIT_ACTUAL"] == 0) & (cc["AMT_BALANCE"] > 0)
cc["CC_IS_OVERDRAFT"] = overdraft_mask.astype("int8")
cc.loc[overdraft_mask, "CC_UTILIZATION"] = 1.0
print(f"Filas límite=0 con saldo deudor: {overdraft_mask.sum():,}")

# Indicadores binarios
cc["CC_HAS_DPD"] = (cc["SK_DPD"] > 0).astype("int8")
cc["CC_HAS_DRAWING_ATM"] = (cc["AMT_DRAWINGS_ATM_CURRENT"].fillna(0) > 0).astype("int8")

print(f"\n=== SK_DPD ===")
print(f"Filas con SK_DPD > 0: {(cc['SK_DPD'] > 0).sum():,} "
      f"({(cc['SK_DPD'] > 0).mean()*100:.3f}%)")
print(f"\n=== Avances en cajero (ATM) ===")
print(f"Filas con drawing ATM > 0: {cc['CC_HAS_DRAWING_ATM'].sum():,} "
      f"({cc['CC_HAS_DRAWING_ATM'].mean()*100:.2f}%)")

Filas límite=0 con saldo deudor: 7,992

=== SK_DPD ===
Filas con SK_DPD > 0: 153,355 (3.993%)

=== Avances en cajero (ATM) ===
Filas con drawing ATM > 0: 424,777 (11.06%)


In [6]:
agg_dict = {
    "MONTHS_BALANCE": ["count", "min"],
    "AMT_BALANCE": ["mean", "max"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean", "max"],
    "CC_UTILIZATION": ["mean", "max"],
    "CC_IS_OVERDRAFT": ["max", "mean", "sum"],
    "AMT_DRAWINGS_CURRENT": ["mean", "sum", "max"],
    "AMT_DRAWINGS_ATM_CURRENT": ["sum"],
    "AMT_PAYMENT_CURRENT": ["mean", "sum"],
    "AMT_TOTAL_RECEIVABLE": ["mean", "max"],
    "CNT_DRAWINGS_CURRENT": ["mean", "sum"],
    "SK_DPD": ["max", "mean"],
    "SK_DPD_DEF": ["max", "mean"],
    "CC_HAS_DPD": ["max", "mean", "sum"],
    "CC_HAS_DRAWING_ATM": ["mean", "sum"],
}

cc_agg = cc.groupby("SK_ID_CURR").agg(agg_dict)
cc_agg.columns = ["CC_" + "_".join(c).upper() for c in cc_agg.columns]
cc_agg = cc_agg.reset_index()

# Conteo de tarjetas distintas
n_cards = cc.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().rename("CC_N_CARDS")
cc_agg = cc_agg.merge(n_cards, on="SK_ID_CURR")

print(f"Shape: {cc_agg.shape}")
print(cc_agg.head(3))

Shape: (103558, 32)
   SK_ID_CURR  CC_MONTHS_BALANCE_COUNT  CC_MONTHS_BALANCE_MIN  \
0      100006                        6                     -6   
1      100011                       74                    -75   
2      100013                       96                    -96   

   CC_AMT_BALANCE_MEAN  CC_AMT_BALANCE_MAX  CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN  \
0             0.000000                0.00                    270000.000000   
1         54482.111149           189000.00                    164189.189189   
2         18159.919219           161420.22                    131718.750000   

   CC_AMT_CREDIT_LIMIT_ACTUAL_MAX  CC_CC_UTILIZATION_MEAN  \
0                          270000                0.000000   
1                          180000                0.302678   
2                          157500                0.115301   

   CC_CC_UTILIZATION_MAX  CC_CC_IS_OVERDRAFT_MAX  CC_CC_IS_OVERDRAFT_MEAN  \
0                0.00000                       0                      0.0   
1  

In [7]:
cc_target = cc_agg.merge(train_target, on="SK_ID_CURR", how="inner")
print(f"Clientes con tarjeta + target: {cc_target.shape}")

cc_features = [c for c in cc_target.columns if c.startswith("CC_")]
auc_list = []

for col in cc_features:
    m = cc_target[col].notna()
    if m.sum() == 0 or cc_target.loc[m, col].nunique() < 2:
        continue
        
    auc_raw = roc_auc_score(cc_target.loc[m, "TARGET"], cc_target.loc[m, col])
    auc_list.append({
        "feature": col,
        "auc": round(max(auc_raw, 1 - auc_raw), 4),
        "direction": "↓ menos default" if auc_raw < 0.5 else "↑ más default",
        "pct_null": round(100 * cc_target[col].isna().mean(), 2),
    })

auc_cc = pd.DataFrame(auc_list).sort_values("auc", ascending=False)
print(f"\n=== Top features de credit_card por AUC ===")
print(auc_cc.to_string(index=False))
auc_cc.to_csv(REPORTS / "auc_rank_credit_card.csv", index=False)

Clientes con tarjeta + target: (86905, 33)

=== Top features de credit_card por AUC ===
                        feature    auc       direction  pct_null
         CC_CC_UTILIZATION_MEAN 0.6292   ↑ más default      0.85
   CC_CNT_DRAWINGS_CURRENT_MEAN 0.6235   ↑ más default      0.00
     CC_CC_HAS_DRAWING_ATM_MEAN 0.6120   ↑ más default      0.00
   CC_AMT_DRAWINGS_CURRENT_MEAN 0.6071   ↑ más default      0.00
          CC_CC_UTILIZATION_MAX 0.6070   ↑ más default      0.85
            CC_AMT_BALANCE_MEAN 0.6051   ↑ más default      0.00
   CC_AMT_TOTAL_RECEIVABLE_MEAN 0.6042   ↑ más default      0.00
    CC_CNT_DRAWINGS_CURRENT_SUM 0.5956   ↑ más default      0.00
             CC_AMT_BALANCE_MAX 0.5821   ↑ más default      0.00
    CC_AMT_TOTAL_RECEIVABLE_MAX 0.5807   ↑ más default      0.00
    CC_AMT_DRAWINGS_CURRENT_MAX 0.5723   ↑ más default      0.00
      CC_CC_HAS_DRAWING_ATM_SUM 0.5654   ↑ más default      0.00
CC_AMT_DRAWINGS_ATM_CURRENT_SUM 0.5644   ↑ más default      0.00
  

In [8]:
cc_agg.to_parquet(PROCESSED / "credit_card_aggregated.parquet", index=False)
mem = cc_agg.memory_usage(deep=True).sum() / 1024**2
print(f"Guardado: credit_card_aggregated.parquet")
print(f"Shape: {cc_agg.shape} | Memoria: {mem:.1f} MB")

del cc, cc_target
gc.collect()

Guardado: credit_card_aggregated.parquet
Shape: (103558, 32) | Memoria: 17.8 MB


10